# GPU Profiling: Base vs Control vs Runtime-Aware

Profiles all three models with the same setup (same hardware, batch size, and input) using PyTorch Profiler.
Fine-tuned adapters are merged into the base model before profiling.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

os.environ.setdefault("HF_HOME", str(Path.home() / ".cache" / "huggingface"))
PROJECT_ROOT = Path(os.environ.get('EFFICIENT_CODEGEN_ROOT', '/workspace/efficient-codegen'))
os.chdir(PROJECT_ROOT)

BASE_MODEL      = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
CONTROL_ADAPTER = str(PROJECT_ROOT / 'checkpoints/control_full')
RUNTIME_ADAPTER = str(PROJECT_ROOT / 'checkpoints/runtime_aware_full')

CONTROL_MERGED  = str(PROJECT_ROOT / 'checkpoints/control_full_merged')
RUNTIME_MERGED  = str(PROJECT_ROOT / 'checkpoints/runtime_aware_full_merged')

PROFILE_INPUT   = 'data/curated/validation/dataset_clean.json'
BATCH_SIZE      = 64
MAX_NEW_TOKENS  = 128
LIMIT           = 320

WANDB_PROJECT   = 'hpml-efficient-codegen'
WANDB_ENTITY    = 'efficient-codegen'
USE_WANDB       = True

print('PROJECT_ROOT:', PROJECT_ROOT)
print('BASE_MODEL:  ', BASE_MODEL)
print('CONTROL_ADAPTER:', CONTROL_ADAPTER)
print('RUNTIME_ADAPTER:', RUNTIME_ADAPTER)
print('USE_WANDB:', USE_WANDB)


PROJECT_ROOT: /workspace/efficient-codegen
BASE_MODEL:   Qwen/Qwen2.5-Coder-1.5B-Instruct
CONTROL_ADAPTER: /workspace/efficient-codegen/checkpoints/control_full
RUNTIME_ADAPTER: /workspace/efficient-codegen/checkpoints/runtime_aware_full
USE_WANDB: True


In [2]:
import wandb
import os
os.environ["HF_HOME"] = "/root/.cache/huggingface"
os.environ["WANDB_API_KEY"] = "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"  # paste your key here, or set it as an env var
if os.environ.get("WANDB_API_KEY"):
    wandb.login(key=os.environ["WANDB_API_KEY"])
else:
    os.environ.setdefault("WANDB_MODE", "disabled")
    USE_WANDB = False
    print("WANDB_API_KEY not set — W&B logging disabled.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yz3202 (efficient-codegen) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Step 1: Merge LoRA Adapters

Fine-tuned checkpoints are LoRA adapters — merge them into the base model weights before profiling.
Skip if merged checkpoints already exist.

In [3]:
def merge_adapter(adapter_path, output_dir):
    if Path(output_dir).exists():
        print(f'Already merged: {output_dir}')
        return
    print(f'Merging {adapter_path} -> {output_dir}')
    subprocess.run([
        sys.executable, 'serving/merge_checkpoint.py',
        '--adapter_path',    adapter_path,
        '--base_model_name', BASE_MODEL,
        '--output_dir',      output_dir,
    ], check=True)
    print('Done.')

merge_adapter(CONTROL_ADAPTER, CONTROL_MERGED)
merge_adapter(RUNTIME_ADAPTER, RUNTIME_MERGED)

Merging /workspace/efficient-codegen/checkpoints/control_full -> /workspace/efficient-codegen/checkpoints/control_full_merged
Loading base model: Qwen/Qwen2.5-Coder-1.5B-Instruct


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 1219.58it/s]


Loading adapter: /workspace/efficient-codegen/checkpoints/control_full
Merging adapter weights into base model...
Saving merged model to /workspace/efficient-codegen/checkpoints/control_full_merged


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.58s/it]


Done.
Done.
Merging /workspace/efficient-codegen/checkpoints/runtime_aware_full -> /workspace/efficient-codegen/checkpoints/runtime_aware_full_merged
Loading base model: Qwen/Qwen2.5-Coder-1.5B-Instruct


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 1121.37it/s]


Loading adapter: /workspace/efficient-codegen/checkpoints/runtime_aware_full
Merging adapter weights into base model...
Saving merged model to /workspace/efficient-codegen/checkpoints/runtime_aware_full_merged


Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.59s/it]


Done.
Done.


## Step 2: Profile Each Model

Runs `profile_model.py` for each model with identical settings.
Traces are saved to `outputs/gpu_profiling/<model>/`.

In [4]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Torch: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [5]:
MODELS = {
    'base':          BASE_MODEL,
    'control':       CONTROL_MERGED,
    'runtime_aware': RUNTIME_MERGED,
}

TRACE_BASE = PROJECT_ROOT / 'outputs/gpu_profiling'

summaries = {}

In [7]:
for label, model_path in MODELS.items():
    trace_dir = str(TRACE_BASE / label)
    print(f'\n{"="*60}')
    print(f'Profiling: {label}')
    print(f'{"="*60}')
    cmd = [
        sys.executable, 'profiling/profile_model.py',
        '--input_path',     PROFILE_INPUT,
        '--model_name',     model_path,
        '--trace_dir',      trace_dir,
        '--limit',          str(LIMIT),
        '--batch_size',     str(BATCH_SIZE),
        '--max_new_tokens', str(MAX_NEW_TOKENS),
        '--wandb_project',  WANDB_PROJECT,
        '--wandb_entity',   WANDB_ENTITY,
        '--wandb_run_name', f'gpu-profiling-{label}',
    ]
    if USE_WANDB:
        cmd.append('--use_wandb')
    subprocess.run(cmd, check=True)
    summaries[label] = {'trace_dir': trace_dir}



Profiling: base


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: yz3202 (efficient-codegen) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /workspace/efficient-codegen/wandb/run-20260505_013421-aqgzwhzr
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run gpu-profiling-base
wandb: ⭐️ View project at https://wandb.ai/efficient-codegen/hpml-efficient-codegen
wandb: 🚀 View run at https://wandb.ai/efficient-codegen/hpml-efficient-codegen/runs/aqgzwhzr
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 1143.22it/s]
/workspace/efficient-codegen/profiling/profile_model.py:181: UserWarning: Profiler won't be using warmup, this can skew profiler results
  schedule=schedule(wait=0, warmup=0, active=1),
/venv/main/lib/python3.12/site-packages/torch/profiler/p

[batch 1/5] batch_size=64 input_len=222 output_len=128 latency=6.9923s tok/s=1171.6
[batch 2/5] batch_size=64 input_len=313 output_len=128 latency=6.1393s tok/s=1334.4
[batch 3/5] batch_size=64 input_len=341 output_len=128 latency=6.1819s tok/s=1325.2
[batch 4/5] batch_size=64 input_len=533 output_len=128 latency=6.3914s tok/s=1281.7
[batch 5/5] batch_size=64 input_len=563 output_len=128 latency=6.5178s tok/s=1256.9

Summary:
Average latency:      6.444523219764233
Average input length: 394.4
Average output len:   128.0
Trace directory:      /workspace/efficient-codegen/outputs/gpu_profiling/base
Peak CUDA memory MB:  6248.05


wandb: updating run metadata
wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:       avg_input_len ▁
wandb:       avg_latency_s ▁
wandb:      avg_output_len ▁
wandb: peak_cuda_memory_mb ▁
wandb: 
wandb: Run summary:
wandb:       avg_input_len 394.4
wandb:       avg_latency_s 6.44452
wandb:      avg_output_len 128
wandb: peak_cuda_memory_mb 6248.04541
wandb: 
wandb: 🚀 View run gpu-profiling-base at: https://wandb.ai/efficient-codegen/hpml-efficient-codegen/runs/aqgzwhzr
wandb: ⭐️ View project at: https://wandb.ai/efficient-codegen/hpml-efficient-codegen
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260505_013421-aqgzwhzr/logs



Profiling: control


...

wandb: updating run metadata
wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: uploading data
wandb: 
wandb: Run history:
wandb:       avg_input_len ▁
wandb:       avg_latency_s ▁
wandb:      avg_output_len ▁
wandb: peak_cuda_memory_mb ▁
wandb: 
wandb: Run summary:
wandb:       avg_input_len 394.4
wandb:       avg_latency_s 6.55604
wandb:      avg_output_len 128
wandb: peak_cuda_memory_mb 6248.04541
wandb: 
wandb: 🚀 View run gpu-profiling-control at: https://wandb.ai/efficient-codegen/hpml-efficient-codegen/runs/mc588fgl
wandb: ⭐️ View project at: https://wandb.ai/efficient-codegen/hpml-efficient-codegen
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260505_013627-mc588fgl/logs



Profiling: runtime_aware


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: yz3202 (efficient-codegen) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /workspace/efficient-codegen/wandb/run-20260505_013835-noqg155m
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run gpu-profiling-runtime_aware
wandb: ⭐️ View project at https://wandb.ai/efficient-codegen/hpml-efficient-codegen
wandb: 🚀 View run at https://wandb.ai/efficient-codegen/hpml-efficient-codegen/runs/noqg155m
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 862.67it/s]
/workspace/efficient-codegen/profiling/profile_model.py:181: UserWarning: Profiler won't be using warmup, this can skew profiler results
  schedule=schedule(wait=0, warmup=0, active=1),
/venv/main/lib/python3.12/site-packages/torch/pr

[batch 1/5] batch_size=64 input_len=222 output_len=128 latency=6.5218s tok/s=1256.1
[batch 2/5] batch_size=64 input_len=313 output_len=128 latency=6.1924s tok/s=1322.9
[batch 3/5] batch_size=64 input_len=341 output_len=128 latency=6.1825s tok/s=1325.0
[batch 4/5] batch_size=64 input_len=533 output_len=128 latency=6.4773s tok/s=1264.7
[batch 5/5] batch_size=64 input_len=563 output_len=128 latency=6.9583s tok/s=1177.3

Summary:
Average latency:      6.466479004547
Average input length: 394.4
Average output len:   128.0
Trace directory:      /workspace/efficient-codegen/outputs/gpu_profiling/runtime_aware
Peak CUDA memory MB:  6248.05


wandb: updating run metadata
wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading output.log; uploading wandb-summary.json
wandb: 
wandb: Run history:
wandb:       avg_input_len ▁
wandb:       avg_latency_s ▁
wandb:      avg_output_len ▁
wandb: peak_cuda_memory_mb ▁
wandb: 
wandb: Run summary:
wandb:       avg_input_len 394.4
wandb:       avg_latency_s 6.46648
wandb:      avg_output_len 128
wandb: peak_cuda_memory_mb 6248.04541
wandb: 
wandb: 🚀 View run gpu-profiling-runtime_aware at: https://wandb.ai/efficient-codegen/hpml-efficient-codegen/runs/noqg155m
wandb: ⭐️ View project at: https://wandb.ai/efficient-codegen/hpml-efficient-codegen
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260505_013835-noqg155m/logs


## Step 3: Parse Traces and Compare

In [8]:
import json
from collections import defaultdict
import pandas as pd

TOP_N = 15
REPORT = {}

for label in MODELS:
    trace_dir = TRACE_BASE / label
    trace_files = sorted(trace_dir.rglob('*.pt.trace.json')) if trace_dir.exists() else []
    if not trace_files:
        print(f'No traces found for {label}')
        continue

    trace_file = max(trace_files, key=lambda f: f.stat().st_size)
    print(f'\n[{label}] Parsing: {trace_file.name}')

    with open(trace_file) as f:
        data = json.load(f)

    events = data.get('traceEvents', data) if isinstance(data, dict) else data
    op_time = defaultdict(float)
    op_calls = defaultdict(int)
    total = 0.0
    for e in events:
        if not isinstance(e, dict):
            continue
        if e.get('ph') == 'X' and e.get('dur', 0) > 0:
            name = e.get('name', 'unknown')
            dur_ms = e['dur'] / 1000
            op_time[name] += dur_ms
            op_calls[name] += 1
            total += dur_ms

    top = sorted(op_time.items(), key=lambda x: x[1], reverse=True)[:TOP_N]
    REPORT[label] = {'total_ms': total, 'top_ops': top, 'op_time': op_time, 'op_calls': op_calls}

    print(f"  Total traced time: {total:.2f} ms")
    print(f"  {'Operator':<50} {'ms':>10} {'%':>7}")
    print(f"  {'-'*70}")
    for name, ms in top:
        pct = ms / total * 100
        print(f"  {name:<50} {ms:>10.2f} {pct:>6.1f}%")


[base] Parsing: worker0.1777943989629352468.pt.trace.json
  Total traced time: 38877.08 ms
  Operator                                                   ms       %
  ----------------------------------------------------------------------
  ProfilerStep#4                                       13486.69   34.7%
  PyTorch Profiler (0)                                  6885.20   17.7%
  void at::native::elementwise_kernel<128, 4, at::native::gpu_kernel_impl_nocast<at::native::direct_copy_kernel_cuda(at::TensorIteratorBase&)::{lambda()#3}::operator()() const::{lambda()#12}::operator()() const::{lambda(c10::BFloat16)#1}>(at::TensorIteratorBase&, at::native::direct_copy_kernel_cuda(at::TensorIteratorBase&)::{lambda()#3}::operator()() const::{lambda()#12}::operator()() const::{lambda(c10::BFloat16)#1} const&)::{lambda(int)#1}>(int, at::native::gpu_kernel_impl_nocast<at::native::direct_copy_kernel_cuda(at::TensorIteratorBase&)::{lambda()#3}::operator()() const::{lambda()#12}::operator()() const::{

## Step 4: Side-by-Side Comparison

Top operators compared across all three models.

In [9]:
KEY_OPS = [
    'aten::linear',
    'aten::scaled_dot_product_attention',
    'aten::matmul',
    'aten::mm',
    'aten::addmm',
    'aten::mul',
    'aten::add',
    'aten::item',
    'cudaStreamSynchronize',
    'cudaLaunchKernel',
]

rows = []
for op in KEY_OPS:
    row = {'Operator': op}
    for label in MODELS:
        if label not in REPORT:
            row[f'{label} ms'] = None
            row[f'{label} %'] = None
            continue
        ms = REPORT[label]['op_time'].get(op, 0.0)
        pct = ms / REPORT[label]['total_ms'] * 100
        row[f'{label} ms'] = round(ms, 2)
        row[f'{label} %'] = round(pct, 2)
    rows.append(row)

df = pd.DataFrame(rows).set_index('Operator')
print('Key Operator Comparison (ms and % of total traced time)')
print('=' * 80)
df

Key Operator Comparison (ms and % of total traced time)


,base ms,base %,control ms,control %,runtime_aware ms,runtime_aware %
Operator,,,,,,
aten::linear,1197.11,3.08,1200.70,3.10,1286.43,3.18
aten::scaled_dot_product_attention,593.30,1.53,576.86,1.49,622.09,1.54
aten::matmul,565.34,1.45,573.18,1.48,611.63,1.51
aten::mm,422.06,1.09,435.26,1.13,458.43,1.13
aten::addmm,376.95,0.97,382.78,0.99,405.04,1.00
aten::mul,457.67,1.18,459.38,1.19,503.39,1.24
aten::add,314.95,0.81,311.76,0.81,337.39,0.83
aten::item,841.23,2.16,850.43,2.20,826.14,2.04
cudaStreamSynchronize,837.21,2.15,846.22,2.19,821.95,2.03


## Step 5: TensorBoard

View traces for all three models side by side in TensorBoard.
Each model's traces are in a separate subdirectory, which TensorBoard uses as the run label.

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir outputs/gpu_profiling

## Step 6: Operator-Level Profiling

Runs `profile_operators.py` for each model, producing a bottleneck report and CSV of operator times.
Output is saved to `outputs/operator_profiling/<model>/`.

In [11]:
OP_PROFILE_BASE = PROJECT_ROOT / 'outputs/operator_profiling'

In [14]:
tmp_dir = '/workspace/efficient-codegen/tmp'
os.makedirs(tmp_dir, exist_ok=True)
env = os.environ.copy()
env['TMPDIR'] = tmp_dir

for label, model_path in MODELS.items():
    output_dir = str(OP_PROFILE_BASE / label)
    print(f'\n{"="*60}')
    print(f'Operator profiling: {label}')
    print(f'{"="*60}')
    subprocess.run([
        sys.executable, 'profiling/profile_operators.py',
        '--input_path',     PROFILE_INPUT,
        '--model_name',     model_path,
        '--limit',          str(LIMIT),
        '--batch_size',     str(BATCH_SIZE),
        '--max_new_tokens', str(MAX_NEW_TOKENS),
        '--output_dir',     output_dir,
    ], check=True, env=env)



Operator profiling: base
Device : cuda  |  dtype : torch.bfloat16
Output : /workspace/efficient-codegen/outputs/operator_profiling/base


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 1050.46it/s]



Warming up (1 example, not profiled)...
Warm-up done.

Profiling 320 example(s) in 5 batch(es) of up to 64...



/venv/main/lib/python3.12/site-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


[batch 1/5]  batch_size=64  input_len=222  output_len=128  latency=6.523s  tok/s=1255.8
[batch 2/5]  batch_size=64  input_len=313  output_len=128  latency=6.486s  tok/s=1263.1
[batch 3/5]  batch_size=64  input_len=341  output_len=128  latency=6.513s  tok/s=1257.7
[batch 4/5]  batch_size=64  input_len=533  output_len=128  latency=6.821s  tok/s=1201.1
[batch 5/5]  batch_size=64  input_len=563  output_len=128  latency=7.203s  tok/s=1137.4

Chrome trace saved → /workspace/efficient-codegen/outputs/operator_profiling/base/chrome_trace.json
  View at: ui.perfetto.dev  or  chrome://tracing
TensorBoard trace  → /workspace/efficient-codegen/outputs/operator_profiling/base/tb_trace/worker0.1777946470551.pt.trace.json

Note: CUDA times are zero — reporting CPU times instead. This is normal when CUDA kernels run asynchronously and attribution is unavailable in this PyTorch build.
Operator CSV saved  → /workspace/efficient-codegen/outputs/operator_profiling/base/operators.csv

OPERATOR-LEVEL BOTTLE

...

[batch 1/5]  batch_size=64  input_len=222  output_len=128  latency=6.466s  tok/s=1266.9
[batch 2/5]  batch_size=64  input_len=313  output_len=128  latency=6.558s  tok/s=1249.1
[batch 3/5]  batch_size=64  input_len=341  output_len=128  latency=6.527s  tok/s=1255.0
[batch 4/5]  batch_size=64  input_len=533  output_len=128  latency=6.920s  tok/s=1183.8
[batch 5/5]  batch_size=64  input_len=563  output_len=128  latency=7.465s  tok/s=1097.4

Chrome trace saved → /workspace/efficient-codegen/outputs/operator_profiling/control/chrome_trace.json
  View at: ui.perfetto.dev  or  chrome://tracing
TensorBoard trace  → /workspace/efficient-codegen/outputs/operator_profiling/control/tb_trace/worker0.1777947047674.pt.trace.json

Note: CUDA times are zero — reporting CPU times instead. This is normal when CUDA kernels run asynchronously and attribution is unavailable in this PyTorch build.
Operator CSV saved  → /workspace/efficient-codegen/outputs/operator_profiling/control/operators.csv

OPERATOR-LEV

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 775.32it/s]



Warming up (1 example, not profiled)...
Warm-up done.

Profiling 320 example(s) in 5 batch(es) of up to 64...



/venv/main/lib/python3.12/site-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


[batch 1/5]  batch_size=64  input_len=222  output_len=128  latency=6.551s  tok/s=1250.6
[batch 2/5]  batch_size=64  input_len=313  output_len=128  latency=6.433s  tok/s=1273.5
[batch 3/5]  batch_size=64  input_len=341  output_len=128  latency=6.552s  tok/s=1250.4
[batch 4/5]  batch_size=64  input_len=533  output_len=128  latency=7.016s  tok/s=1167.6
[batch 5/5]  batch_size=64  input_len=563  output_len=128  latency=7.316s  tok/s=1119.8

Chrome trace saved → /workspace/efficient-codegen/outputs/operator_profiling/runtime_aware/chrome_trace.json
  View at: ui.perfetto.dev  or  chrome://tracing
TensorBoard trace  → /workspace/efficient-codegen/outputs/operator_profiling/runtime_aware/tb_trace/worker0.1777947629127.pt.trace.json

Note: CUDA times are zero — reporting CPU times instead. This is normal when CUDA kernels run asynchronously and attribution is unavailable in this PyTorch build.
Operator CSV saved  → /workspace/efficient-codegen/outputs/operator_profiling/runtime_aware/operators

## Step 7: Operator Bottleneck Reports

In [ ]:
for label in MODELS:
    report = OP_PROFILE_BASE / label / 'bottleneck_report.txt'
    if report.exists():
        print(f'\n{"="*60}')
        print(f'  {label}')
        print(f'{"="*60}')
        print(report.read_text(encoding='utf-8')[:3000])
    else:
        print(f'No report found for {label}')



  base
OPERATOR-LEVEL BOTTLENECK REPORT

Top 30 operators by CPU self-time:

Operator                                                   CUDA ms     CPU ms    Calls  % total
-----------------------------------------------------------------------------------------------
full_generate                                                0.000   7389.694        1    27.2%
decode_remaining                                             0.000   6367.912      122    23.4%
cudaLaunchKernel                                             0.000   1577.710   177664     5.8%
aten::linear                                                 0.000   1524.913    25216     5.6%
aten::matmul                                                 0.000    710.307    14592     2.6%
aten::scaled_dot_product_attention                           0.000    682.632     3584     2.5%
aten::mul                                                    0.000    528.391    32768     1.9%
aten::mm                                                  

## Step 8: Operator CSV Comparison

Loads `operators.csv` from each model run and shows a side-by-side comparison of top operators by CPU time.

In [ ]:
import pandas as pd

op_dfs = {}
for label in MODELS:
    csv_path = OP_PROFILE_BASE / label / 'operators.csv'
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        op_dfs[label] = df.set_index('operator') if 'operator' in df.columns else df
    else:
        print(f'No operators.csv for {label}')

if op_dfs:
    col = 'cpu_time_ms'
    top_ops = (
        pd.concat({k: v[col] for k, v in op_dfs.items() if col in v.columns}, axis=1)
        .fillna(0)
        .assign(total=lambda d: d.sum(axis=1))
        .sort_values('total', ascending=False)
        .drop(columns='total')
        .head(20)
    )
    print('Top 20 operators by CPU time (ms)')
    print('=' * 60)
    display(top_ops)


Top 20 operators by CPU time (ms)


,base,control,runtime_aware
operator,,,
full_generate,39622.0127,38515.0279,38663.0004
decode_remaining,32676.2051,31709.1970,31781.3081
cudaLaunchKernel,7749.2191,7406.6470,7421.4346
aten::linear,7672.5207,7410.4473,7434.3708
aten::scaled_dot_product_attention,3573.8853,3491.5352,3515.3066
aten::matmul,3543.9373,3429.9480,3450.0296
aten::item,3018.5527,3029.6504,3036.0915
aten::is_nonzero,3018.1456,3029.2698,3035.8988
aten::_local_scalar_dense,3014.1759,3025.3426,3031.5609
